In [ ]:
import os
import numpy as np
import pandas as pd
import torch

from tqdm.auto import tqdm
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
DATA_RAW_DIR = os.path.join(PROJECT_ROOT, "data", "raw")
DATA_PROCESSED_DIR = os.path.join(PROJECT_ROOT, "data", "processed")
MODELS_DIR = os.path.join(PROJECT_ROOT, "models")
OUTPUTS_DIR = os.path.join(PROJECT_ROOT, "outputs")
TABLES_DIR = os.path.join(OUTPUTS_DIR, "tables")

os.makedirs(TABLES_DIR, exist_ok=True)

FAKE_LABEL = 1
REAL_LABEL = 0

best_dir = os.path.join(MODELS_DIR, "bert_fulltext_trunc256")

train_df = pd.read_csv(os.path.join(DATA_PROCESSED_DIR, "train.csv"), usecols=["full_text", "label"])
test_df = pd.read_csv(os.path.join(DATA_PROCESSED_DIR, "test.csv"), usecols=["full_text", "label"])

test_df["full_text"] = test_df["full_text"].fillna("").astype(str)
test_df["label"] = test_df["label"].astype(int)

raw_csvs = [f for f in os.listdir(DATA_RAW_DIR) if f.lower().endswith(".csv")]
raw_csvs = sorted(raw_csvs, key=lambda f: os.path.getsize(os.path.join(DATA_RAW_DIR, f)), reverse=True)
if len(raw_csvs) == 0:
    raise FileNotFoundError(f"No .csv found in {DATA_RAW_DIR}")
RAW_PATH = os.path.join(DATA_RAW_DIR, raw_csvs[0])

try:
    raw_df = pd.read_csv(RAW_PATH)
except Exception:
    raw_df = pd.read_csv(RAW_PATH, encoding_errors="replace")

def _col(df, name):
    for c in df.columns:
        if c.lower() == name:
            return c
    raise KeyError(name)

title_col = _col(raw_df, "title")
text_col = _col(raw_df, "text")

raw_df[title_col] = raw_df[title_col].fillna("").astype(str)
raw_df[text_col] = raw_df[text_col].fillna("").astype(str)
raw_df["full_text"] = (raw_df[title_col] + " " + raw_df[text_col]).str.replace(r"\s+", " ", regex=True).str.strip()
raw_df = raw_df.drop_duplicates(subset=["full_text"], keep="first").reset_index(drop=True)
raw_df["title"] = raw_df[title_col].str.replace(r"\s+", " ", regex=True).str.strip()
raw_df["text"] = raw_df[text_col].str.replace(r"\s+", " ", regex=True).str.strip()

test_df = test_df.merge(raw_df[["full_text", "title", "text"]], on="full_text", how="left")
test_df["title"] = test_df["title"].fillna("")
test_df["text"] = test_df["text"].fillna("")
test_df["text_snippet"] = test_df["text"].astype(str).str.slice(0, 400)
test_df["full_text_snippet"] = test_df["full_text"].astype(str).str.slice(0, 400)

def save_error_tables(df, model_name, score_col, out_prefix, n=25):
    fp = df[(df["label"] == REAL_LABEL) & (df["pred"] == FAKE_LABEL)].copy()
    fn = df[(df["label"] == FAKE_LABEL) & (df["pred"] == REAL_LABEL)].copy()

    fp = fp.sort_values(score_col, ascending=False).head(n)
    fn = fn.sort_values(score_col, ascending=True).head(n)

    fp_out = fp[["label", "pred", score_col, "title", "text_snippet", "full_text_snippet"]].copy()
    fn_out = fn[["label", "pred", score_col, "title", "text_snippet", "full_text_snippet"]].copy()

    fp_out.insert(0, "model", model_name)
    fp_out.insert(1, "error_type", "false_positive")
    fn_out.insert(0, "model", model_name)
    fn_out.insert(1, "error_type", "false_negative")

    out_fp_path = os.path.join(TABLES_DIR, f"{out_prefix}_fp_top{n}.csv")
    out_fn_path = os.path.join(TABLES_DIR, f"{out_prefix}_fn_top{n}.csv")

    fp_out.to_csv(out_fp_path, index=False, encoding="utf-8-sig")
    fn_out.to_csv(out_fn_path, index=False, encoding="utf-8-sig")

    print("Saved:", out_fp_path)
    print("Saved:", out_fn_path)

    combined = pd.concat([fp_out, fn_out], ignore_index=True)
    out_combined_path = os.path.join(TABLES_DIR, f"{out_prefix}_fp_fn_top{n}.csv")
    combined.to_csv(out_combined_path, index=False, encoding="utf-8-sig")
    print("Saved:", out_combined_path)

    return out_fp_path, out_fn_path, out_combined_path

X_train = train_df["full_text"].fillna("").astype(str).tolist()
y_train = train_df["label"].astype(int).to_numpy()
X_test = test_df["full_text"].tolist()

vectorizer = TfidfVectorizer(
    lowercase=True,
    stop_words="english",
    ngram_range=(1, 2),
    max_features=50000,
    min_df=2,
    max_df=0.95,
    sublinear_tf=True
)

Xtr = vectorizer.fit_transform(X_train)
Xte = vectorizer.transform(X_test)

svm = LinearSVC(class_weight="balanced")
svm.fit(Xtr, y_train)

svm_pred = svm.predict(Xte)
svm_score = svm.decision_function(Xte)

tfidf_df = test_df.copy()
tfidf_df["pred"] = svm_pred.astype(int)
tfidf_df["score_fake"] = svm_score.astype(float)

save_error_tables(tfidf_df, "TFIDF_LinearSVC", "score_fake", "error_analysis_tfidf_linearsvc", n=25)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
tokenizer = AutoTokenizer.from_pretrained(best_dir, use_fast=True)
model = AutoModelForSequenceClassification.from_pretrained(best_dir).to(device)
model.eval()

MAX_LEN = 256
BATCH_SIZE = 16

class InferDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len):
        self.texts = [str(t) for t in texts]
        self.labels = [int(y) for y in labels]
        self.tokenizer = tokenizer
        self.max_len = int(max_len)

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        enc = self.tokenizer(
            self.texts[idx],
            truncation=True,
            padding="max_length",
            max_length=self.max_len,
            return_tensors="pt"
        )
        return {
            "input_ids": enc["input_ids"].squeeze(0),
            "attention_mask": enc["attention_mask"].squeeze(0),
            "label": torch.tensor(self.labels[idx], dtype=torch.long)
        }

loader = DataLoader(
    InferDataset(test_df["full_text"].tolist(), test_df["label"].tolist(), tokenizer, MAX_LEN),
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0
)

all_probs = []
all_preds = []

with torch.no_grad():
    for batch in tqdm(loader, desc="BERT inference"):
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        logits = model(input_ids=input_ids, attention_mask=attention_mask).logits
        probs = torch.softmax(logits, dim=1).detach().cpu().numpy()
        preds = np.argmax(probs, axis=1)
        all_probs.append(probs)
        all_preds.append(preds)

all_probs = np.concatenate(all_probs, axis=0)
all_preds = np.concatenate(all_preds, axis=0)

bert_df = test_df.copy()
bert_df["pred"] = all_preds.astype(int)
bert_df["prob_fake"] = all_probs[:, FAKE_LABEL].astype(float)

save_error_tables(bert_df, "BERT_trunc256", "prob_fake", "error_analysis_bert_trunc256", n=25)